### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
!pip uninstall -y langchain langchain-community langchain-openai langchain-core langsmith langchain-huggingface
!pip install --upgrade --force-reinstall langchain==0.2.0 langchain-community==0.2.0 langchain-openai==0.1.7 langchain-core==0.2.0 chromadb==0.4.24

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.1
Uninstalling langchain-core-1.3.1:
  Successfully uninstalled langchain-core-1.3.1
Found existing installation: langsmith 0.7.34
Uninstalling langsmith-0.7.34:
  Successfully uninstalled langsmith-0.7.34
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 6.2 MB/s eta 0:00:00
INFO: pip

In [2]:
import langchain
print(f"LangChain version: {langchain.__version__}")
import langchain_core
print(f"LangChain Core version: {langchain_core.__version__}")
import langchain_community
print(f"LangChain Community version: {langchain_community.__version__}")

LangChain version: 0.2.0
LangChain Core version: 0.2.0
LangChain Community version: 0.2.0


In [4]:
## langchain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
import os

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

### 1. Sample Data

In [5]:
#include you files

### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "/home/ contents",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")


Loaded 2 documents

First document preview:
Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has b...


In [7]:
documents

[Document(page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.', metadata={'source': '/home/ contents/python.txt'}),
 Document(page_content='Skip to main content\nGoogle Classroom\nClassroom\nSAIT Python LLM Training\nHome\nCalendar\nEnrolled\nTo-do\nS\nSAIT Python LLM Training\nS\nSAIT (PYTHON BATCH)\nPYTHON - A\nC\nCPP Aurobindo College\nArchived classes\nSettings\nStream\nClasswork\nPeople\nNo topic\nMaterial\nbook\nAdversarial Prompting Demo\nPosted 12:20\u202fPM\nMaterial\nbook\nOpenAI API Key 13/05/2026\nPosted 12:19\u202fPM\

### Document Splitting

In [8]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,  # Maximum size of each chunk
    chunk_overlap=30,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 12 chunks from 2 documents

Chunk example:
Content: Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido v...
Metadata: {'source': '/home/ contents/python.txt'}


In [9]:
#Review Chunks
chunks

[Document(page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and', metadata={'source': '/home/ contents/python.txt'}),
 Document(page_content='Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.', metadata={'source': '/home/ contents/python.txt'}),
 Document(page_content='Skip to main content\nGoogle Classroom\nClassroom\nSAIT Python LLM Training\nHome\nCalendar\nEnrolled\nTo-do\nS\nSAIT Python LLM Training\nS\nSAIT (PYTHON BATCH)\nPYTHON - A\nC\nCPP Aurobindo College\nArchived classes\nSettings\nStream\nClasswork\nPeople\nNo topic\nMaterial\nbook\nAdver

### Embedding Models- OpenAI or Hugging Face

In [10]:
#Optional Content-with Open AI embedding Model
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata
embeddings=OpenAIEmbeddings(model="text-embedding-3-small",api_key=userdata.get('OPENAI_API_KEY'))

In [11]:
### Huggingface And OpenAI Models

# Using HuggingFaceEmbeddings from langchain_community which is compatible with the upgraded langchain stack.
from langchain_community.embeddings import HuggingFaceEmbeddings

## Initialize a simple Embedding model (no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langsmith-0.8.3-py3-none-any.whl.metadata (15 kB)
Using cached langchain_core-1.4.0-py3-none-any.whl (548 kB)
Using cached langsmith-0.8.3-py3-none-any.whl (399 kB)
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.52
    Uninstalling langchain-core-0.1.52:
      Successfully uninstalled langchain-core-0.1.52
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.1.7 requires langchain-core<0.3,>=0.1.46, but you have langchain-core 1.4.0 which is incompatible.
langchain 0.1.20 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.4.0 which is

ImportError: cannot import name 'ModelProfile' from 'langchain_core.language_models' (/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/__init__.py)

In [1]:
#Vectors produced by the Model

#Check embedding model with sample text
sample_text="Machine LEarning is fascinating"
vector=embeddings.embed_query(sample_text)
vector

NameError: name 'embeddings' is not defined

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [ ]:
#Available Splitted chunks
chunks

In [ ]:
## Create a Chromdb vector store
from langchain_community.vectorstores import Chroma
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"

)
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

### Test Similarity Search

In [ ]:
query="What are the applications of ML ?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=1)
similar_docs

In [ ]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

### Advanced Similarity Search With Scores

In [ ]:
results_scores=vectorstore.similarity_search_with_score(query,k=5)
results_scores

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

llm=ChatOpenAI(
    model_name="gpt-3.5-turbo",
    api_key=userdata.get('OPENAI_API_KEY')
)


In [ ]:
test_response=llm.invoke("What is Large Language Models")
test_response

### Modern RAG Chain

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

In [ ]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [ ]:
prompt

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [ ]:
### Create a document chain
from langchain.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [ ]:
### Create The Final RAG Chain
from langchain.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

In [ ]:
response=rag_chain.invoke({"input":"What are the reasons for climate change ?"})

In [ ]:
response

In [ ]:
response['answer']

In [ ]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})

    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?",
    "What is capital of USA ?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [ ]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [ ]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

In [ ]:
retriever

In [ ]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

In [ ]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

In [ ]:
retriever.get_relevant_documents("What is Deep Learning")

In [ ]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [ ]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

In [ ]:
query_rag_lcel("What is machine learning?")

In [ ]:
query_rag_lcel("what is Q-Networks ?")

### Add New Documents To Existing Vector Store

In [ ]:
vectorstore

In [ ]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
in RL include: states, actions, rewards, policies, and value functions. Popular RL
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [ ]:
new_document

In [ ]:
chunks

In [ ]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [ ]:
new_doc

In [ ]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

In [ ]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)



In [ ]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

In [ ]:
## query with the updated vector
new_question="what is Q-Networks ?"
result=query_rag_lcel(new_question)
result

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [ ]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

In [ ]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational RAG chain created!")

In [ ]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [ ]:
chat_history

In [ ]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

In [ ]:
result2['answer']

### Using GROQ LLM's


In [ ]:
llm

In [ ]:
load_dotenv()

In [ ]:
os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [ ]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
llm=ChatGroq(model="gemma2-9b-it",api_key=os.getenv("GROQ_API_KEY"))
llm

In [ ]:
llm=init_chat_model(model="groq:gemma2-9b-it")
llm